In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))  

In [2]:
from data.loader import load_bars, load_multi_tf

df_h1 = load_bars("GBPUSD", "H1")
tfs = load_multi_tf("GBPUSD", ["D1", "H4", "H1", "M15"])

In [3]:
tfs["D1"].head()

,datetime,open,high,low,close,volume,spread
0,2012-01-11 00:00:00+00:00,1.54617,1.54875,1.53090,1.53335,205023.11,0.0
1,2012-01-12 00:00:00+00:00,1.53335,1.53672,1.52790,1.53401,228509.90,0.0
2,2012-01-13 00:00:00+00:00,1.53401,1.54093,1.52336,1.53163,231252.52,0.0
3,2012-01-14 00:00:00+00:00,1.53163,1.53163,1.53163,1.53163,0.00,0.0
4,2012-01-15 00:00:00+00:00,1.53163,1.53163,1.52749,1.53009,5436.31,0.0


# 1 · General preprocessing (all timeframes)

Applied identically to every TF before per-TF feature engineering:

1. Dedup + sort by timestamp
2. Drop bad ticks (OHLC inconsistent, zero/negative volume — weekend dead bars from resample)
3. Clip return outliers (fat-finger ticks, rolling z-score)
4. Gap-fill on regular grid, capped run length
5. Tag train/val/test split boundary (no lookahead downstream)

In [4]:
import numpy as np
import pandas as pd

TF_FREQ = {"M15": "15min", "H1": "1h", "H4": "4h", "D1": "1D"}
MAX_FFILL_BARS = 4
OUTLIER_SIGMA = 8.0


def dedup_and_sort(df: pd.DataFrame) -> pd.DataFrame:
    """Drop duplicate timestamps (keep last), sort ascending."""
    df = df.drop_duplicates(subset="datetime", keep="last")
    return df.sort_values("datetime").reset_index(drop=True)


def filter_bad_ticks(df: pd.DataFrame) -> pd.DataFrame:
    """Drop bars with inconsistent OHLC or non-positive volume (weekend dead bars, glitches)."""
    valid = (
        (df["high"] >= df["low"])
        & (df["high"] >= df["open"])
        & (df["high"] >= df["close"])
        & (df["low"] <= df["open"])
        & (df["low"] <= df["close"])
        & (df["volume"] > 0)
    )
    return df[valid].reset_index(drop=True)


def clip_price_outliers(df: pd.DataFrame, sigma: float = OUTLIER_SIGMA) -> pd.DataFrame:
    """Drop close-to-close returns beyond `sigma` rolling std (fat-finger ticks)."""
    ret = df["close"].pct_change()
    roll_std = ret.rolling(200, min_periods=50).std()
    z = (ret / roll_std).abs()
    bad = (z > sigma).fillna(False)
    return df[~bad].reset_index(drop=True)


def fill_gaps(df: pd.DataFrame, timeframe: str, max_ffill: int = MAX_FFILL_BARS) -> pd.DataFrame:
    """
    Reindex to a regular grid for `timeframe`, forward-fill short gaps only.
    Bars still missing after `max_ffill` consecutive steps are dropped rather
    than silently carried forward indefinitely (stale-data guard).
    """
    freq = TF_FREQ[timeframe]
    full_index = pd.date_range(df["datetime"].iloc[0], df["datetime"].iloc[-1], freq=freq)
    reindexed = df.set_index("datetime").reindex(full_index)
    reindexed.index.name = "datetime"

    was_missing = reindexed["close"].isna()
    filled = reindexed.copy()
    filled[["open", "high", "low", "close"]] = filled[["open", "high", "low", "close"]].ffill(limit=max_ffill)
    filled["volume"] = filled["volume"].fillna(0.0)
    if "spread" in filled.columns:
        filled["spread"] = filled["spread"].ffill(limit=max_ffill)

    # synthetic filled bars: O=H=L=C=prior close, volume=0 (flags a "no trade" bar downstream)
    synth = was_missing & filled["close"].notna()
    for col in ("open", "high", "low"):
        filled.loc[synth, col] = filled.loc[synth, "close"]

    filled = filled.dropna(subset=["open", "high", "low", "close"])
    return filled.reset_index().rename(columns={"index": "datetime"})


def tag_splits(df: pd.DataFrame, train_end: str, val_end: str) -> pd.DataFrame:
    """Tag each row's split membership so downstream fit-on-train-only steps have a boundary."""
    train_end_ts = pd.Timestamp(train_end, tz="UTC")
    val_end_ts = pd.Timestamp(val_end, tz="UTC")
    df = df.copy()
    df["split"] = np.select(
        [df["datetime"] <= train_end_ts, df["datetime"] <= val_end_ts],
        ["train", "val"],
        default="test",
    )
    return df


def preprocess(
    df: pd.DataFrame,
    timeframe: str,
    train_end: str | None = None,
    val_end: str | None = None,
    max_ffill: int = MAX_FFILL_BARS,
    outlier_sigma: float = OUTLIER_SIGMA,
) -> pd.DataFrame:
    """Full general preprocessing pipeline for one timeframe's OHLCV DataFrame."""
    df = dedup_and_sort(df)
    df = filter_bad_ticks(df)
    df = clip_price_outliers(df, sigma=outlier_sigma)
    df = fill_gaps(df, timeframe, max_ffill=max_ffill)
    if train_end is not None and val_end is not None:
        df = tag_splits(df, train_end, val_end)
    return df

In [5]:
# split boundaries — adjust to your walk-forward scheme
TRAIN_END = "2022-12-31"
VAL_END = "2024-12-31"

clean = {
    tf: preprocess(df, tf, train_end=TRAIN_END, val_end=VAL_END)
    for tf, df in tfs.items()
}

for tf, df in clean.items():
    n_raw, n_clean = len(tfs[tf]), len(df)
    print(
        f"{tf}: {n_raw} -> {n_clean} bars  "
        f"({df['datetime'].iloc[0]} .. {df['datetime'].iloc[-1]})  "
        f"dropped={n_raw - n_clean}  "
        f"split={df['split'].value_counts().to_dict()}"
    )

D1: 4654 -> 5295 bars  (2012-01-11 00:00:00+00:00 .. 2026-07-10 00:00:00+00:00)  dropped=-641  split={'train': 4008, 'val': 731, 'test': 556}
H4: 24672 -> 26431 bars  (2012-01-11 00:00:00+00:00 .. 2026-07-10 00:00:00+00:00)  dropped=-1759  split={'train': 20006, 'val': 3644, 'test': 2781}
H1: 96094 -> 93510 bars  (2012-01-11 01:00:00+00:00 .. 2026-07-10 00:00:00+00:00)  dropped=2584  split={'train': 70814, 'val': 12877, 'test': 9819}
M15: 384171 -> 364489 bars  (2012-01-11 01:30:00+00:00 .. 2026-07-10 00:00:00+00:00)  dropped=19682  split={'train': 275977, 'val': 50224, 'test': 38288}


In [6]:
from config.settings import DATA_PROCESSED

SYMBOL = "GBPUSD"

for tf, df in clean.items():
    path = DATA_PROCESSED / f"{SYMBOL}_{tf}.parquet"
    df.to_parquet(path, index=False)
    print(f"[SAVE] {path.relative_to(DATA_PROCESSED.parent.parent)}: {len(df)} rows")

[SAVE] data/processed/GBPUSD_D1.parquet: 5295 rows
[SAVE] data/processed/GBPUSD_H4.parquet: 26431 rows
[SAVE] data/processed/GBPUSD_H1.parquet: 93510 rows
[SAVE] data/processed/GBPUSD_M15.parquet: 364489 rows
